# Phase 1: Layout and Handwriting Test

This notebook sends a chaotic document image to a vision-capable OpenAI model using the raw SDK, with the image embedded as a base64 data URL and `detail="high"`.

In [1]:
%pip install -q openai pillow pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import json
import pandas as pd

from vision_lab_utils import (
    build_response_record,
    call_vision_json,
    ensure_dir,
    get_client,
    save_json,
)

client = get_client()
ROOT = Path.cwd()
OUTPUT_DIR = ensure_dir(ROOT / "outputs")
IMAGE_PATH = ROOT / "ChatGPT Image Apr 24, 2026, 02_56_34 PM.png"
MODEL = "gpt-4.1-mini"
DETAIL = "high"

assert IMAGE_PATH.exists(), f"Image not found: {IMAGE_PATH}"

In [3]:
EXTRACTION_PROMPT = """
You are a document understanding assistant.

Analyze the attached image carefully. The image may contain:
- multiple text columns
- one or more structured tables
- handwritten notes that may be messy, slanted, or misspelled

Your tasks:
1. Extract every table as valid JSON.
   - Use the visible column headers as keys.
   - Return each row as a separate object in an array.
   - Preserve values exactly as written.
2. Extract multi-column printed text while preserving reading separation by column.
   - Return each column independently as column_1, column_2, etc.
   - Do not merge text from different columns.
3. Extract all handwritten text separately.
   - First transcribe exactly as written.
   - If there are spelling mistakes, add a corrected version in brackets.
4. Do not summarize.
5. If something is unclear, mark it as [unclear].

Return only valid JSON in this exact structure:
{
  \"table\": [...],
  \"columns\": {
    \"column_1\": \"...\",
    \"column_2\": \"...\"
  },
  \"handwritten\": {
    \"raw\": \"...\",
    \"corrected\": \"...\"
  }
}
""".strip()

print(EXTRACTION_PROMPT)

You are a document understanding assistant.

Analyze the attached image carefully. The image may contain:
- multiple text columns
- one or more structured tables
- handwritten notes that may be messy, slanted, or misspelled

Your tasks:
1. Extract every table as valid JSON.
   - Use the visible column headers as keys.
   - Return each row as a separate object in an array.
   - Preserve values exactly as written.
2. Extract multi-column printed text while preserving reading separation by column.
   - Return each column independently as column_1, column_2, etc.
   - Do not merge text from different columns.
3. Extract all handwritten text separately.
   - First transcribe exactly as written.
   - If there are spelling mistakes, add a corrected version in brackets.
4. Do not summarize.
5. If something is unclear, mark it as [unclear].

Return only valid JSON in this exact structure:
{
  "table": [...],
  "columns": {
    "column_1": "...",
    "column_2": "..."
  },
  "handwritten": {
   

In [4]:
parsed, response = call_vision_json(
    client,
    model=MODEL,
    prompt=EXTRACTION_PROMPT,
    image_path=IMAGE_PATH,
    detail=DETAIL,
)

record = build_response_record(
    model=MODEL,
    detail=DETAIL,
    image_path=IMAGE_PATH,
    prompt=EXTRACTION_PROMPT,
    parsed=parsed,
    response=response,
)

save_json(OUTPUT_DIR / "phase1_extraction.json", record)
print(json.dumps(parsed, indent=2, ensure_ascii=False))
record["usage"]

{
  "table": [
    {
      "Date": "01-Apr-24",
      "Donor Name": "EcoMart",
      "Category": "Corporate",
      "Amount ($)": "1,500.00",
      "Payment Method": "Bank Transfer"
    },
    {
      "Date": "05-Apr-24",
      "Donor Name": "Rahul Verma",
      "Category": "Individual",
      "Amount ($)": "250.00",
      "Payment Method": "UPI"
    },
    {
      "Date": "07-Apr-24",
      "Donor Name": "Green Future Ltd.",
      "Category": "Corporate",
      "Amount ($)": "2,000.00",
      "Payment Method": "Bank Transfer"
    },
    {
      "Date": "12-Apr-24",
      "Donor Name": "Sneha Patel",
      "Category": "Individual",
      "Amount ($)": "100.00",
      "Payment Method": "UPI"
    },
    {
      "Date": "15-Apr-24",
      "Donor Name": "Planet Lovers Club",
      "Category": "NGO",
      "Amount ($)": "750.00",
      "Payment Method": "Bank Transfer"
    },
    {
      "Date": "18-Apr-24",
      "Donor Name": "Anonymous",
      "Category": "Individual",
      "Amount ($)"

{'input_tokens': 2740,
 'output_tokens': 961,
 'total_tokens': 3701,
 'input_tokens_details': {'cached_tokens': 0},
 'output_tokens_details': {'reasoning_tokens': 0}}

In [5]:
table_df = pd.DataFrame(parsed.get("table", []))
table_df

,Date,Donor Name,Category,Amount ($),Payment Method
0,01-Apr-24,EcoMart,Corporate,"1,500.00",Bank Transfer
1,05-Apr-24,Rahul Verma,Individual,250.00,UPI
2,07-Apr-24,Green Future Ltd.,Corporate,"2,000.00",Bank Transfer
3,12-Apr-24,Sneha Patel,Individual,100.00,UPI
4,15-Apr-24,Planet Lovers Club,NGO,750.00,Bank Transfer
5,18-Apr-24,Anonymous,Individual,50.00,UPI
6,22-Apr-24,Kiran Enterprises,Corporate,"3,000.00",Bank Transfer
7,28-Apr-24,Amit Joshi,Individual,200.00,UPI


## Reflection Questions

- Prompt used: We used a zero-shot prompt that clearly told the model to extract tables as JSON, keep multi-column text separated by column, and return handwritten text in both raw and corrected form.

- Actual prompt used:

```text
You are a document understanding assistant.

Analyze the attached image carefully. The image may contain:
- multiple text columns
- one or more structured tables
- handwritten notes that may be messy, slanted, or misspelled

Your tasks:
1. Extract every table as valid JSON.
   - Use the visible column headers as keys.
   - Return each row as a separate object in an array.
   - Preserve values exactly as written.
2. Extract multi-column printed text while preserving reading separation by column.
   - Return each column independently as column_1, column_2, etc.
   - Do not merge text from different columns.
3. Extract all handwritten text separately.
   - First transcribe exactly as written.
   - If there are spelling mistakes, add a corrected version in brackets.
4. Do not summarize.
5. If something is unclear, mark it as [unclear].

Return only valid JSON in this exact structure:
{
  "table": [...],
  "columns": {
    "column_1": "...",
    "column_2": "..."
  },
  "handwritten": {
    "raw": "...",
    "corrected": "..."
  }
}
```

- Handwriting transcription: The model mostly copied the handwritten text exactly as written. In this run, the raw and corrected outputs were almost identical, which suggests that the model did not make many corrections and preserved the handwriting closely.

- Multi-column handling: The model handled the multi-column layout well. It correctly identified the document as having three separate columns and returned them independently:

column_1: THE GREEN PLANET INITIATIVE
column_2: A GREENER FUTURE STARTS TODAY
column_3: VOLUNTEER STORIES

It kept the text from each column separate and did not mix content between columns.